# 01 · Tratamento e preparação dos dados

**Projeto:** NPS Preditivo, Tech Challenge Fase 1 (POSTECH Data Science + IA, FIAP)

**Objetivo deste notebook:** conhecer a base, verificar a qualidade dos dados, documentar inconsistências e gerar uma base tratada (`data/processed/nps_tratado.csv`) pronta para a análise exploratória.

**Princípio adotado:** nenhum registro é excluído. Com 2.500 pedidos e inconsistências que atingem grupos inteiros, remover linhas distorceria a análise. As inconsistências recebem uma *flag* e são tratadas onde importam.

In [1]:
# Bibliotecas e funções do projeto
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Permite importar o módulo src/ a partir da pasta notebooks/
sys.path.append(str(Path.cwd().parent))
from src.nps_utils import (
    CAMINHO_BASE_TRATADA,
    calcular_nps,
    carregar_base_bruta,
    classificar_nps,
)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.2f}".format)

## 1. Carga da base e visão geral

In [2]:
df_bruto = carregar_base_bruta()
print(f"Linhas: {df_bruto.shape[0]:,} | Colunas: {df_bruto.shape[1]}")
df_bruto.head()

Linhas: 2,500 | Colunas: 19

,customer_id,customer_age,customer_region,customer_tenure_months,order_id,order_value,items_quantity,discount_value,payment_installments,delivery_time_days,delivery_delay_days,freight_value,delivery_attempts,customer_service_contacts,resolution_time_days,nps_score,repeat_purchase_30d,complaints_count,csat_internal_score
0,1,63,Nordeste,14,50001,139.73,4,39.35,4,2,2,55.53,3,0,4,6.90,0,3,6.50
1,2,20,Sul,1,50002,458.95,2,9.51,10,6,4,28.23,3,0,10,2.40,0,3,0.00
2,3,46,Nordeste,111,50003,507.06,5,42.82,6,6,1,40.99,1,4,5,4.80,0,7,1.50
3,4,52,Centro-Oeste,117,50004,302.19,2,19.58,9,5,2,35.24,3,1,11,5.90,0,4,0.30
4,5,56,Norte,50,50005,253.06,1,29.37,11,13,1,39.32,1,1,0,6.10,0,3,7.90


In [3]:
# Tipos de dados de cada coluna
df_bruto.dtypes.to_frame("tipo")

,tipo
customer_id,int64
customer_age,int64
customer_region,str
customer_tenure_months,int64
order_id,int64
order_value,float64
items_quantity,int64
discount_value,float64
payment_installments,int64
delivery_time_days,int64


## 2. Verificações de qualidade

### 2.1 Valores ausentes e duplicidades

In [4]:
print("Valores ausentes por coluna (total):", int(df_bruto.isna().sum().sum()))
print("Linhas totalmente duplicadas:", df_bruto.duplicated().sum())
print("order_id repetidos:", df_bruto["order_id"].duplicated().sum())
print("customer_id repetidos:", df_bruto["customer_id"].duplicated().sum())

Valores ausentes por coluna (total): 0
Linhas totalmente duplicadas: 0
order_id repetidos: 0
customer_id repetidos: 0

**Leitura:** a base não tem valores ausentes nem duplicados. Cada cliente aparece **uma única vez**, então nesta base *cliente = pedido*. Não há histórico de compras por cliente, o que limita análises de comportamento ao longo do tempo.

### 2.2 Faixas de valores (estatística descritiva)

In [5]:
df_bruto.drop(columns=["customer_id", "order_id"]).describe().T

,count,mean,std,min,25%,50%,75%,max
customer_age,2500.00,43.40,14.89,18.00,31.00,43.00,56.00,69.00
customer_tenure_months,2500.00,61.32,34.48,1.00,31.00,62.00,91.00,119.00
order_value,2500.00,434.26,289.77,7.76,220.25,375.51,577.29,1983.81
items_quantity,2500.00,3.47,1.69,1.00,2.00,3.00,5.00,6.00
discount_value,2500.00,29.75,29.23,0.02,8.88,20.94,40.83,230.33
payment_installments,2500.00,6.00,3.16,1.00,3.00,6.00,9.00,11.00
delivery_time_days,2500.00,8.02,3.77,2.00,5.00,8.00,11.00,14.00
delivery_delay_days,2500.00,2.19,1.45,0.00,1.00,2.00,3.00,8.00
freight_value,2500.00,38.22,12.08,2.62,29.93,38.50,46.27,76.13
delivery_attempts,2500.00,2.01,0.82,1.00,1.00,2.00,3.00,3.00


In [6]:
# Distribuição das regiões
df_bruto["customer_region"].value_counts().to_frame("pedidos")

,pedidos
customer_region,
Sul,521
Sudeste,520
Norte,506
Nordeste,485
Centro-Oeste,468


**Leitura:** todos os valores estão em faixas plausíveis (idade de 18 a 69 anos, nota de 0 a 10, atraso de 0 a 8 dias, sem valores negativos). As cinco regiões têm volumes parecidos, entre 468 e 521 pedidos.

### 2.3 Escala da nota de NPS

Na metodologia original a pergunta do NPS aceita **apenas números inteiros** de 0 a 10. Vamos verificar se a base segue esse padrão.

In [7]:
nota_inteira = (df_bruto["nps_score"] == df_bruto["nps_score"].round()).mean()
print(f"Notas inteiras: {nota_inteira:.0%} | Notas com casas decimais: {1 - nota_inteira:.0%}")
df_bruto["nps_score"].sample(8, random_state=1).to_frame()

Notas inteiras: 17% | Notas com casas decimais: 83%

,nps_score
1406,4.20
297,0.60
1276,2.10
2368,0.00
438,6.30
2009,5.20
642,5.60
655,8.30


**Achado:** 83% das notas têm casas decimais (ex.: 6.9, 4.8). Isso foge do padrão da pesquisa de NPS e sugere que a nota foi transformada, gerada por simulação ou é uma média.

**Decisão:** manter a nota original e classificar com limites contínuos que respeitam a régua oficial:

| Categoria | Régua oficial | Regra aplicada nesta base |
|---|---|---|
| Detrator | 0 a 6 | nota < 7 |
| Neutro | 7 e 8 | 7 ≤ nota < 9 |
| Promotor | 9 e 10 | nota ≥ 9 |

### 2.4 Consistência entre colunas

Algumas colunas deveriam "conversar" entre si. Testamos as regras de negócio esperadas:

In [8]:
checagens = pd.DataFrame({
    "regra_testada": [
        "Cliente sem contato com o atendimento, mas com tempo de resolução > 0",
        "Dias de atraso maiores que o tempo total de entrega",
        "Desconto maior que o valor do pedido",
        "Frete maior que o valor do pedido",
        "Mais reclamações do que contatos com o atendimento",
    ],
    "pedidos_afetados": [
        ((df_bruto["customer_service_contacts"] == 0) & (df_bruto["resolution_time_days"] > 0)).sum(),
        (df_bruto["delivery_delay_days"] > df_bruto["delivery_time_days"]).sum(),
        (df_bruto["discount_value"] > df_bruto["order_value"]).sum(),
        (df_bruto["freight_value"] > df_bruto["order_value"]).sum(),
        (df_bruto["complaints_count"] > df_bruto["customer_service_contacts"]).sum(),
    ],
})
checagens["pct_da_base"] = (checagens["pedidos_afetados"] / len(df_bruto) * 100).round(1)
checagens

,regra_testada,pedidos_afetados,pct_da_base
0,"Cliente sem contato com o atendimento, mas com tempo de resolução > 0",511,20.40
1,Dias de atraso maiores que o tempo total de entrega,121,4.80
2,Desconto maior que o valor do pedido,35,1.40
3,Frete maior que o valor do pedido,18,0.70
4,Mais reclamações do que contatos com o atendimento,2428,97.10


**Interpretação e tratamento de cada inconsistência:**

1. **Tempo de resolução sem contato (511 de 554 clientes sem contato).** Não faz sentido ter tempo de resolução se o cliente nunca acionou o atendimento. Criamos `tempo_resolucao_valido`, que fica vazio quando não houve contato, e analisamos o tempo de resolução **apenas entre quem acionou o atendimento**.
2. **Atraso maior que o tempo total (121 pedidos).** Pode indicar que `delivery_time_days` registra o *prazo prometido*, e não o tempo real. Os pedidos são mantidos e marcados com uma flag.
3. **Desconto maior que o valor do pedido (35 pedidos).** É provável que `order_value` já seja o valor líquido, depois do desconto. Os pedidos são mantidos e marcados com uma flag.
4. **Frete maior que o valor do pedido (18 pedidos).** É plausível em itens baratos. Os pedidos são mantidos.
5. **Mais reclamações que contatos (97% da base).** Indica que `complaints_count` deve somar reclamações de vários canais (Reclame Aqui, redes sociais, marketplace) ou do histórico do cliente, e não só do pedido. Recomendamos confirmar a definição com a área dona do dado.

Esses pontos entram na seção de **limitações** do relatório e dos slides.

### 2.5 Valores extremos (outliers)

Usamos a regra do intervalo interquartil (IQR) para contar valores fora do padrão nas colunas de valores monetários.

In [9]:
def contar_outliers_iqr(serie: pd.Series) -> int:
    """Conta valores acima de Q3 + 1,5*IQR ou abaixo de Q1 - 1,5*IQR."""
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    return int(((serie < q1 - 1.5 * iqr) | (serie > q3 + 1.5 * iqr)).sum())

colunas_monetarias = ["order_value", "discount_value", "freight_value"]
pd.DataFrame({
    "outliers_iqr": [contar_outliers_iqr(df_bruto[c]) for c in colunas_monetarias],
    "maximo": [df_bruto[c].max() for c in colunas_monetarias],
}, index=colunas_monetarias)

,outliers_iqr,maximo
order_value,84,1983.81
discount_value,125,230.33
freight_value,12,76.13


**Decisão:** os valores extremos são pedidos de alto valor, que existem de fato no e-commerce, e não erros de digitação. Por isso **são mantidos**. Nas análises usamos faixas (ex.: "Acima de R$800"), que não se distorcem com extremos.

## 3. Classificação das variáveis pelo momento da jornada

Este é o cuidado mais importante para o objetivo do negócio: **antecipar** o NPS. Uma variável só serve para isso se estiver disponível **antes** da pesquisa.

In [10]:
momento_jornada = pd.DataFrame([
    ("customer_age, customer_region, customer_tenure_months", "Cadastro", "Antes da compra", "Sim"),
    ("order_value, items_quantity, discount_value, payment_installments, freight_value", "Pedido", "Na compra", "Sim"),
    ("delivery_time_days, delivery_delay_days, delivery_attempts", "Logística", "Durante a entrega", "Sim"),
    ("customer_service_contacts, resolution_time_days, complaints_count", "Atendimento", "Durante/após a entrega", "Sim, com atenção ao horário do registro"),
    ("csat_internal_score", "Satisfação interna", "Mesmo momento ou após o NPS", "Não: outra medida de satisfação"),
    ("repeat_purchase_30d", "Resultado", "Até 30 dias APÓS o pedido", "Não: é consequência, não causa"),
    ("nps_score", "Alvo", "Fim da jornada", "É o que queremos antecipar"),
], columns=["variaveis", "grupo", "quando_existe", "serve_para_antecipar_o_nps"])
momento_jornada

,variaveis,grupo,quando_existe,serve_para_antecipar_o_nps
0,"customer_age, customer_region, customer_tenure_months",Cadastro,Antes da compra,Sim
1,"order_value, items_quantity, discount_value, payment_installments, freight_value",Pedido,Na compra,Sim
2,"delivery_time_days, delivery_delay_days, delivery_attempts",Logística,Durante a entrega,Sim
3,"customer_service_contacts, resolution_time_days, complaints_count",Atendimento,Durante/após a entrega,"Sim, com atenção ao horário do registro"
4,csat_internal_score,Satisfação interna,Mesmo momento ou após o NPS,Não: outra medida de satisfação
5,repeat_purchase_30d,Resultado,Até 30 dias APÓS o pedido,"Não: é consequência, não causa"
6,nps_score,Alvo,Fim da jornada,É o que queremos antecipar


In [11]:
# Evidência de que repeat_purchase_30d é consequência da satisfação (vazamento)
df_bruto.assign(categoria=df_bruto["nps_score"].apply(classificar_nps)) \
    .groupby("categoria")["repeat_purchase_30d"].mean().mul(100).round(1) \
    .to_frame("pct_que_recomprou_em_30d")

,pct_que_recomprou_em_30d
categoria,
Detrator,0.00
Neutro,38.40
Promotor,100.00


**Achado:** 100% dos promotores recompraram e nenhum detrator recomprou. Uma relação tão perfeita mostra que `repeat_purchase_30d` é **resultado** da satisfação. Ela ajuda a mostrar *por que o NPS importa*, mas não pode explicar nem antecipar o NPS. O mesmo vale para `csat_internal_score`, que é outra forma de medir a mesma satisfação.

## 4. Criação de variáveis para a análise

In [12]:
df = df_bruto.copy()

# --- Alvo em categorias de NPS ---
df["nps_categoria"] = df["nps_score"].apply(classificar_nps)
df["is_detrator"] = (df["nps_categoria"] == "Detrator").astype(int)
df["is_promotor"] = (df["nps_categoria"] == "Promotor").astype(int)

# --- Indicadores simples de "algo deu errado" ---
df["teve_atraso"] = (df["delivery_delay_days"] > 0).astype(int)
df["teve_contato_atendimento"] = (df["customer_service_contacts"] > 0).astype(int)

# --- Tempo de resolução só faz sentido para quem acionou o atendimento ---
df["tempo_resolucao_valido"] = df["resolution_time_days"].where(df["teve_contato_atendimento"] == 1)

# --- Percentual de desconto sobre o valor do pedido ---
df["pct_desconto"] = (df["discount_value"] / df["order_value"] * 100).round(1)

# --- Flags de inconsistência (documentadas na seção 2.4) ---
df["flag_resolucao_sem_contato"] = ((df["customer_service_contacts"] == 0) & (df["resolution_time_days"] > 0)).astype(int)
df["flag_atraso_maior_que_prazo"] = (df["delivery_delay_days"] > df["delivery_time_days"]).astype(int)
df["flag_desconto_maior_que_valor"] = (df["discount_value"] > df["order_value"]).astype(int)

In [13]:
# --- Faixas em linguagem de negócio (facilitam a leitura por gestores) ---
df["faixa_atraso"] = pd.cut(df["delivery_delay_days"], bins=[-1, 0, 1, 2, 3, 99],
                            labels=["Sem atraso", "1 dia", "2 dias", "3 dias", "4+ dias"])
df["faixa_contatos"] = pd.cut(df["customer_service_contacts"], bins=[-1, 0, 1, 2, 99],
                              labels=["Nenhum", "1 contato", "2 contatos", "3+ contatos"])
df["faixa_reclamacoes"] = pd.cut(df["complaints_count"], bins=[-1, 1, 3, 5, 99],
                                 labels=["0-1", "2-3", "4-5", "6+"])
df["faixa_resolucao"] = pd.cut(df["tempo_resolucao_valido"], bins=[-1, 2, 5, 8, 99],
                               labels=["0-2 dias", "3-5 dias", "6-8 dias", "9-11 dias"])
df["faixa_idade"] = pd.cut(df["customer_age"], bins=[17, 24, 34, 44, 54, 120],
                           labels=["18-24", "25-34", "35-44", "45-54", "55+"])
df["faixa_tempo_casa"] = pd.cut(df["customer_tenure_months"], bins=[0, 12, 36, 60, 999],
                                labels=["Até 1 ano", "1-3 anos", "3-5 anos", "5+ anos"])
df["faixa_valor_pedido"] = pd.cut(df["order_value"], bins=[0, 150, 300, 500, 800, 1e9],
                                  labels=["Até R$150", "R$150-300", "R$300-500", "R$500-800", "Acima de R$800"])

df[["delivery_delay_days", "faixa_atraso", "customer_service_contacts", "faixa_contatos", "nps_score", "nps_categoria"]].head()

,delivery_delay_days,faixa_atraso,customer_service_contacts,faixa_contatos,nps_score,nps_categoria
0,2,2 dias,0,Nenhum,6.90,Detrator
1,4,4+ dias,0,Nenhum,2.40,Detrator
2,1,1 dia,4,3+ contatos,4.80,Detrator
3,2,2 dias,1,1 contato,5.90,Detrator
4,1,1 dia,1,1 contato,6.10,Detrator


## 5. Primeira leitura do alvo

In [14]:
distribuicao = df["nps_categoria"].value_counts(normalize=True).reindex(["Detrator", "Neutro", "Promotor"]).mul(100).round(1)
print(distribuicao.to_string())
print(f"\nNPS da base: {calcular_nps(df['nps_categoria'])}")
print(f"Nota média: {df['nps_score'].mean():.2f}")

nps_categoria
Detrator   84.40
Neutro     11.20
Promotor    4.40

NPS da base: -80.0
Nota média: 4.38

**Leitura:** o NPS da base é **−80**, com 84% de detratores. É um resultado muito abaixo do que se observa no varejo brasileiro, onde o NPS costuma ficar positivo. A comparação com benchmarks de mercado deve ser feita com cautela, já que a escala da nota foge do padrão (seção 2.3). Para a análise, o que importa é **comparar grupos dentro da base**: o que separa clientes satisfeitos dos insatisfeitos.

## 6. Exportação da base tratada

In [15]:
CAMINHO_BASE_TRATADA.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(CAMINHO_BASE_TRATADA, index=False)
print(f"Base tratada salva em: data/processed/{CAMINHO_BASE_TRATADA.name}")
print(f"Linhas: {df.shape[0]:,} | Colunas: {df.shape[1]} ({df.shape[1] - df_bruto.shape[1]} novas)")

Base tratada salva em: data/processed/nps_tratado.csv
Linhas: 2,500 | Colunas: 36 (17 novas)

## Resumo do tratamento

| Etapa | Resultado |
|---|---|
| Ausentes e duplicados | Nenhum encontrado |
| Escala do NPS | Nota decimal; classificada com limites < 7, 7 a < 9 e ≥ 9 |
| Inconsistências | 4 regras violadas, todas marcadas com flag, nenhuma linha removida |
| Tempo de resolução | Considerado só para quem teve contato (`tempo_resolucao_valido`) |
| Outliers | Mantidos (pedidos reais de alto valor) |
| Vazamento | `repeat_purchase_30d` e `csat_internal_score` não servem para antecipar o NPS |
| Novas variáveis | Categoria NPS, flags, faixas em linguagem de negócio, % de desconto |

Próximo passo: `02_analise_exploratoria.ipynb`.